# Chapter 6: Testing jev.decideAll

This notebook is the live layer of testing for `jev.decideAll`, the batch procedure built in chapter 6. It takes a list of items and the same questions for all of them, runs the calls on a small thread pool and returns one row per item. The chapter 5 notebook, `05_decide_tests`, tests the single-item function. This one tests what the batch adds: ordering, per-row errors and concurrency.

Most of the batch behavior can be tested against a closed port, because a failing call still has to come back in the right place. An optional section tests the hosted Jev API and times a sweep of the `concurrency` option.

A few design choices are worth knowing about:

- **Nothing is written to the database.** Every test is a read-only `CALL`.
- **The same statuses as the chapter 5 notebook.** `PASS` and `FAIL` are checks, `INFO` is a measurement and `SKIPPED` means a test couldn't run.
- **The hosted API is off by default.** `RUN_REAL_JEV` is `False`. When it's `True`, the notebook makes 168 calls. Change `REAL_ITEMS`, `REAL_CONCURRENCY` and `REAL_REPEATS` to make that bigger or smaller.
- **Timings are repeated.** A single timing is noisy, so each setting runs several times, in rotation, and we report the minimum, median and maximum.
- **Saving merges into the existing file.** Each test replaces its own entry in `results_decideall.json` and the rest are kept. For a clean snapshot, delete the file before a full run.

## Prerequisites

Set the following environment variables before launching Jupyter. The URI is the local Neo4j Desktop one:

```shell
export NEO4J_URI="bolt://localhost:7687"
export NEO4J_USERNAME="neo4j"
export NEO4J_PASSWORD="your-password"
export NEO4J_DATABASE="neo4j"
```

You also need:

- The plugin JAR in the `plugins` folder of your DBMS, with `dbms.security.procedures.allowlist=jev.*` in `neo4j.conf`.
- For the hosted API only, `server.jvm.additional=-DTYPESAFE_API_KEY=<your-api-key>` in `neo4j.conf`. The notebook never sees the key.

## 1. Install Dependencies

In [1]:
%pip install ipywidgets==8.1.5 \
             jupyterlab-widgets==3.0.13 \
             neo4j==5.28.1 \
             pandas==2.2.3 \
             plotly==6.1.2 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


## 2. Imports

In [2]:
import json
import os
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import time

from datetime import datetime
from neo4j import GraphDatabase
from neo4j.exceptions import Neo4jError
from pathlib import Path

pio.renderers.default = "notebook"

## 3. Configuration

`RUN_REAL_JEV` decides whether the optional hosted-API section runs. Leave it `False` unless the key is in `neo4j.conf` and you're happy to spend the calls. The hosted sweep makes `REAL_ITEMS` calls for every combination of concurrency level and repeat, plus a small batch first: with the defaults, 20 x 4 x 2 + 8 = 168 calls.

`QUESTIONS` holds one choice question, so each item costs one call.

In [3]:
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]

print("Credentials set.")

Credentials set.


In [4]:
RUN_REAL_JEV     = False          # True: also test the hosted Jev API
REAL_ITEMS       = 20             # items per batch in the hosted sweep
REAL_CONCURRENCY = [1, 4, 8, 16]  # concurrency levels in the hosted sweep
REAL_REPEATS     = 2              # repeats of each level in the hosted sweep

CLOSED_URL  = "http://127.0.0.1:9/v1/systemone"   # nothing listens on port 9

RESULTS_FILE = "results_decideall.json"

CATEGORIES = ["retail", "travel", "atm_withdrawal", "fuel"]

QUESTIONS = {
    "decision": {
        "type": "choice",
        "instructions": "Should this transaction be flagged for review or passed through?",
        "criteria": {
            "Flag": "Potential fraud. Send for review.",
            "Pass": "Likely legitimate. Process normally.",
        },
    },
}

CLOSED = {"url": CLOSED_URL, "max_retries": 0}

print(f"RUN_REAL_JEV = {RUN_REAL_JEV}")
print("Configuration set.")

RUN_REAL_JEV = False
Configuration set.


## 4. Connect to Neo4j

In [5]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth = (NEO4J_USERNAME, NEO4J_PASSWORD)
)

print(driver.verify_connectivity())  # None is expected
print("Connection created.")

None
Connection created.


## 5. Test Helpers

`run_all` makes one `jev.decideAll` call and returns the rows as dictionaries, with the wall-clock time in milliseconds. `make_items` builds a list of items with ids like `txn-007` and states that differ from each other, so the items are not all the same request.

`summarize_sweep`, `sweep_figure` and `repeat_figure` turn the repeated timings of a concurrency sweep into a table and two charts. The table has these columns for each level:

- `min`, `median` and `max` are the wall times of the repeats, in milliseconds.
- `speedup` is the median at the lowest level divided by the median at this level.
- `paired_speedup` compares each run with the lowest-level run from the same repeat and takes the median of those ratios. A slow stretch that hits every level in a repeat cancels out, so it's the steadier of the two speedups.
- `spread` is the slowest repeat divided by the fastest one at this level. A value near 1 means steady timings. A value of 2 or more means the speed varied a lot between repeats, so read the speedups with care.

`save_results` merges into the existing file by test name, so a partial re-run updates its own entries and keeps the rest.

In [6]:
RESULTS = []   # one entry per test
TIMINGS = {}   # raw numbers, saved next to the results

def record(name, status, detail=""):
    """Store a result. Running a test again replaces its earlier entry."""
    RESULTS[:] = [r for r in RESULTS if r["test"] != name]
    RESULTS.append({"test": name, "status": status, "detail": str(detail)})
    print(f"{status:8} {name}" + (f" -- {detail}" if detail != "" else ""))

def check(name, condition, detail=""):
    record(name, "PASS" if condition else "FAIL", detail)

def observe(name, detail):
    record(name, "INFO", detail)

def skip(name, reason):
    record(name, "SKIPPED", reason)

def starts(text, prefix):
    return isinstance(text, str) and text.startswith(prefix)

DECIDE_ALL_QUERY = """
CALL jev.decideAll($items, $questions, $options)
YIELD id, answers, error_message, metadata
RETURN id, answers, error_message, metadata
"""

def make_items(n, prefix="txn"):
    return [
        {"id": f"{prefix}-{i:03d}",
         "state": {"amount": round(5 + i * 13.7, 2),
                   "merchant_category": CATEGORIES[i % len(CATEGORIES)],
                   "distance_from_home_km": round((i * 7.3) % 300, 1),
                   "hour_of_day": i % 24}}
        for i in range(n)
    ]

def run_all(items, questions=None, options=None):
    """One jev.decideAll call. Returns the rows and the wall-clock milliseconds."""
    params = dict(
        items=items,
        questions=QUESTIONS if questions is None else questions,
        options=options or {},
    )
    with driver.session(database=NEO4J_DATABASE) as session:
        started = time.perf_counter()
        rows = session.run(DECIDE_ALL_QUERY, **params).data()
        elapsed = (time.perf_counter() - started) * 1000
    return rows, elapsed

def summarize_sweep(sweep_rows):
    df = pd.DataFrame(sweep_rows)
    base = df["concurrency"].min()
    baseline = df[df["concurrency"] == base].set_index("repeat")["wall_ms"]
    df["paired"] = [baseline[repeat] / ms for repeat, ms in zip(df["repeat"], df["wall_ms"])]

    by_level = df.groupby("concurrency")
    out = by_level["wall_ms"].agg(["min", "median", "max"]).round(0)
    out["errors"] = by_level["errors"].sum()
    out["speedup"] = (out.loc[base, "median"] / out["median"]).round(2)
    out["paired_speedup"] = by_level["paired"].median().round(2)
    out["spread"] = (out["max"] / out["min"]).round(2)
    return out.reset_index()

def repeat_figure(sweep_rows, title):
    """One line per repeat on a log scale, to show whether a slow repeat was slow at every level."""
    df = pd.DataFrame(sweep_rows)
    fig = go.Figure()
    for repeat, group in df.groupby("repeat"):
        group = group.sort_values("concurrency")
        fig.add_trace(go.Scatter(x=group["concurrency"].astype(str), y=group["wall_ms"],
                                 mode="lines+markers", name=f"repeat {repeat}", opacity=0.6))
    fig.update_layout(title=title + " (log scale)", xaxis_title="Concurrency",
                      yaxis_title="Wall time for the batch (ms)", yaxis_type="log", height=450)
    return fig

def sweep_figure(summary, title):
    fig = go.Figure(go.Scatter(
        x=summary["concurrency"].astype(str),
        y=summary["median"],
        mode="lines+markers",
        error_y=dict(type="data", symmetric=False,
                     array=summary["max"] - summary["median"],
                     arrayminus=summary["median"] - summary["min"]),
    ))
    fig.update_layout(title=title, xaxis_title="Concurrency",
                      yaxis_title="Wall time for the batch (ms), median with min and max",
                      height=450, yaxis=dict(range=[0, None]))
    return fig

def save_results():
    """Write the results file. An existing file is merged by test name, so a partial re-run
    updates its own entries and keeps the rest."""
    path = Path(RESULTS_FILE)
    merged, timings = {}, {}
    if path.exists():
        try:
            previous = json.loads(path.read_text())
            # A SKIPPED row only means a test couldn't run at that time, so we don't keep it.
            merged = {r["test"]: r for r in previous.get("results", []) if r.get("status") != "SKIPPED"}
            timings = previous.get("timings", {})
        except (ValueError, KeyError, TypeError):
            print(f"Could not read the existing {RESULTS_FILE}. Starting a new file.")
    for r in RESULTS:
        merged[r["test"]] = r
    timings.update(TIMINGS)

    payload = {
        "notebook":  "06_decideall_tests",
        "saved_at":  datetime.now().isoformat(timespec="seconds"),
        "results":   list(merged.values()),
        "timings":   timings,
    }
    path.write_text(json.dumps(payload, indent=2, default=str))
    print(f"Saved {len(merged)} results to {RESULTS_FILE} ({len(RESULTS)} from this session)")

print("Helpers defined.")

Helpers defined.


## 6. The Tests

### The procedure is installed

If this fails, the JAR isn't in the `plugins` folder, the database wasn't restarted or the allowlist line is missing.

In [7]:
with driver.session(database=NEO4J_DATABASE) as session:
    names = [r["name"] for r in session.run(
        "SHOW PROCEDURES YIELD name WHERE name STARTS WITH 'jev' RETURN name"
    )]

check("procedure jev.decideAll is registered", "jev.decideAll" in names, ", ".join(names))

PASS     procedure jev.decideAll is registered -- jev.decideAll


### Rows come back in input order, with their ids

The calls finish in any order, but the rows must come back in the order the items were given, each with its own id. A closed port is enough to test this, because every call fails and still has to land in the right row. We send 20 items with a concurrency of 8.

In [8]:
items = make_items(20)
rows, ms = run_all(items, options={**CLOSED, "concurrency": 8})

check("order: one row per item", len(rows) == 20, f"{len(rows)} rows")
check("order: ids come back in input order", [r["id"] for r in rows] == [i["id"] for i in items])
check("order: every row reports the connection error",
      all(starts(r["error_message"], "io: ConnectException reaching 127.0.0.1:9") for r in rows))
check("order: no row carries the key", all(r["metadata"]["authenticated"] is False for r in rows))

PASS     order: one row per item -- 20 rows
PASS     order: ids come back in input order
PASS     order: every row reports the connection error
PASS     order: no row carries the key


### A bad concurrency value is reported on every row

`concurrency` must be a whole number from 1 to 16. Each bad value below should give every row a `validation:` error that mentions `concurrency`, with zero attempts. The closed port is only a safety net, so a validation failure can never reach a real server.

In [9]:
for bad in [0, 17, "two", 2.5]:
    rows, ms = run_all(make_items(3), options={**CLOSED, "concurrency": bad})
    ok = (len(rows) == 3
          and all(starts(r["error_message"], "validation:") and "concurrency" in r["error_message"]
                  for r in rows)
          and all(r["metadata"]["attempts"] == 0 for r in rows))
    check(f"concurrency {bad!r} is rejected on every row", ok, rows[0]["error_message"] if rows else "")

PASS     concurrency 0 is rejected on every row -- validation: concurrency must be between 1 and 16
PASS     concurrency 17 is rejected on every row -- validation: concurrency must be between 1 and 16
PASS     concurrency 'two' is rejected on every row -- validation: concurrency must be a whole number
PASS     concurrency 2.5 is rejected on every row -- validation: concurrency must be a whole number


### A failing row doesn't stop the others

An item without a `state` fails by itself, and the item after it still runs. Items without an `id` get their position (0, 1, 2 and so on) instead. A `null` item is also reported as a row, if Neo4j lets us send one.

In [10]:
rows, ms = run_all([{"id": "no_state"}, {"id": "has_state", "state": "x"}], options=CLOSED)
check("per-row: a missing state fails only that row",
      starts(rows[0]["error_message"], "validation:") and "needs a state" in rows[0]["error_message"],
      rows[0]["error_message"])
check("per-row: the next item still ran", starts(rows[1]["error_message"], "io:"),
      rows[1]["error_message"])

rows, ms = run_all([{"state": "x"}, {"state": "y"}], options=CLOSED)
check("ids: a missing id falls back to the position", [r["id"] for r in rows] == [0, 1],
      [r["id"] for r in rows])

try:
    rows, ms = run_all([{"id": "a", "state": "x"}, None, {"id": "c", "state": "z"}], options=CLOSED)
    check("null item: reported as a row error",
          len(rows) == 3 and starts(rows[1]["error_message"], "validation:") and rows[1]["id"] == 1,
          f"{len(rows)} rows")
except Neo4jError as exc:
    observe("null item", f"Neo4j rejected the list before the procedure ran: {exc.message[:100]}")

PASS     per-row: a missing state fails only that row -- validation: item 0 needs a state
PASS     per-row: the next item still ran -- io: ConnectException reaching 127.0.0.1:9 (after 1 attempt)
PASS     ids: a missing id falls back to the position -- [0, 1]
PASS     null item: reported as a row error -- 3 rows


### Limits

A batch holds at most 10000 items. One more should be rejected with a message that says so. An empty list should give no rows. A misspelled option should be reported on every row, as in the chapter 5 notebook.

In [11]:
try:
    run_all(make_items(10001), options=CLOSED)
    check("limit: 10001 items are rejected", False, "no error was raised")
except Neo4jError as exc:
    check("limit: 10001 items are rejected, and the message says why", "10000" in exc.message,
          exc.message[:120])

rows, ms = run_all([], options=CLOSED)
check("limit: an empty list gives no rows", rows == [], f"{len(rows)} rows")

rows, ms = run_all(make_items(3), options={**CLOSED, "modle": "x"})
check("options: an unknown name is reported on every row",
      len(rows) == 3 and all(starts(r["error_message"], "validation:") for r in rows),
      rows[0]["error_message"] if rows else "")

PASS     limit: 10001 items are rejected, and the message says why -- Failed to invoke procedure `jev.decideAll`: Caused by: java.lang.IllegalArgumentException: a batch holds at most 10000 i
PASS     limit: an empty list gives no rows -- 0 rows
PASS     options: an unknown name is reported on every row -- validation: unknown option(s): modle; known options are [connect_timeout_ms, max_retries, model, request_timeout_ms, url]


### Optional: the hosted Jev API

Runs only when `RUN_REAL_JEV` is `True`. It needs `TYPESAFE_API_KEY` in `neo4j.conf`. First a small batch checks that the rows are clean and the key was sent. Then the sweep times `REAL_ITEMS` items at each level in `REAL_CONCURRENCY`, repeated `REAL_REPEATS` times and taken in rotation, so a slow minute on the network doesn't land on one setting.

In [12]:
if RUN_REAL_JEV:
    rows, ms = run_all(make_items(8), options={"concurrency": 4})
    check("real Jev: no row errors", all(r["error_message"] is None for r in rows),
          next((r["error_message"] for r in rows if r["error_message"]), ""))
    check("real Jev: the key was sent on every row", all(r["metadata"]["authenticated"] for r in rows))
    check("real Jev: ids come back in order",
          [r["id"] for r in rows] == [i["id"] for i in make_items(8)])

    sweep_rows = []
    for repeat in range(REAL_REPEATS):
        for level in REAL_CONCURRENCY:
            rows, ms = run_all(make_items(REAL_ITEMS), QUESTIONS, {"concurrency": level})
            sweep_rows.append({"concurrency": level, "repeat": repeat, "wall_ms": ms,
                               "errors": sum(r["error_message"] is not None for r in rows)})

    real_summary = summarize_sweep(sweep_rows)
    TIMINGS["real_jev_sweep"] = sweep_rows
    check("real Jev sweep: no errors", int(real_summary["errors"].sum()) == 0,
          f"{int(real_summary['errors'].sum())} errors")
    observe("real Jev sweep: speedup at each level",
            ", ".join(f"{int(c)}: {s}x" for c, s in zip(real_summary["concurrency"], real_summary["speedup"])))
    observe("real Jev sweep: paired speedup at each level",
            ", ".join(f"{int(c)}: {s}x" for c, s in zip(real_summary["concurrency"], real_summary["paired_speedup"])))
    display(real_summary)
    sweep_figure(real_summary, "jev.decideAll against the hosted API").show()
    repeat_figure(sweep_rows, "Every repeat, hosted API").show()
else:
    skip("real Jev", "RUN_REAL_JEV is False")
    print("RUN_REAL_JEV is False -- no calls made to the hosted API.")

SKIPPED  real Jev -- RUN_REAL_JEV is False
RUN_REAL_JEV is False -- no calls made to the hosted API.


## 7. Summary

Every test with its status. A `FAIL` row names what went wrong in the detail column. `INFO` rows hold the measurements, and `SKIPPED` rows show what couldn't run and why.

In [13]:
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.max_rows", 200)

results_df = pd.DataFrame(RESULTS)
print(results_df["status"].value_counts().to_string())
print()
failed = results_df[results_df["status"] == "FAIL"]
print("No failures." if failed.empty else f"{len(failed)} failure(s):")
results_df

status
PASS       16
SKIPPED     1

No failures.


,test,status,detail
0,procedure jev.decideAll is registered,PASS,jev.decideAll
1,order: one row per item,PASS,20 rows
2,order: ids come back in input order,PASS,
3,order: every row reports the connection error,PASS,
4,order: no row carries the key,PASS,
5,concurrency 0 is rejected on every row,PASS,validation: concurrency must be between 1 and 16
6,concurrency 17 is rejected on every row,PASS,validation: concurrency must be between 1 and 16
7,concurrency 'two' is rejected on every row,PASS,validation: concurrency must be a whole number
8,concurrency 2.5 is rejected on every row,PASS,validation: concurrency must be a whole number
9,per-row: a missing state fails only that row,PASS,validation: item 0 needs a state


## 8. Teardown

In [14]:
save_results()
driver.close()
print("Connection closed.")

Saved 17 results to results_decideall.json (17 from this session)
Connection closed.
